In [1]:
# imports

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window, hilbert
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import importlib
import utils as ut
importlib.reload(ut)

<module 'utils' from '/Users/itamarmerfeld/Projects/UATR/dev/utils.py'>

In [2]:
# output
output_folder = '../results/plots'
to_show = True
to_save = True
plot_title = 'DPV experiment 600m'
output_file_name = 'dpv_600m_detection_scores_plot_for_paper'

# SpectrogramParameters
nperseg = 8000
overlap = 0.25
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9

# S2G Parameters
laplacian_quantization_levels = 5
wasserstein_quantization_levels = 3

# F0 Parameters
f = 632  # Hz

### LOAD DATA

In [3]:
# DPV data
src_folder = '../data/croatia/2407_1_600m'
file_list = os.listdir(src_folder)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list[6:-15]:
    _data, fs = librosa.load(os.path.join(src_folder, file))
    data.append(_data)
data = np.concatenate(data)
print("completed loading data")

completed loading data


In [4]:
# GPX
gpx_file = os.path.join(src_folder, '600m_route.gpx')
gpx_data = gpxpy.parse(open(gpx_file))

# get the first track
track = gpx_data.tracks[0]

# get the first segment
segment = track.segments[0]

rx_location = gpxpy.gpx.GPXWaypoint(44.330008, 14.697141)

distances = []
for point in segment.points:
    distance = rx_location.distance_2d(point)
    distances.append(distance)

distances = np.array(distances)
t_distances = np.linspace(0, len(data)/fs, len(distances))
print("completed loading GPS data")

completed loading GPS data


In [5]:
# BG Noise WAVs
noise_src_folder = '../data/croatia/2507_1_1000m'
file_list = os.listdir(noise_src_folder)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

bg_data = []
for file in file_list[:5]:
    _data, fs = librosa.load(os.path.join(noise_src_folder, file))
    bg_data.append(_data)
bg_data = np.concatenate(bg_data)
print(f"bg_data final shape: {bg_data.shape} of duration: {len(bg_data)/fs} seconds")

# Add BG Noise to data
noise_factor = 10
bg_in_shape = np.concatenate([bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data, bg_data])
noisy_data = data + noise_factor * bg_in_shape[:len(data)]
print("completed adding noise to data")

bg_data final shape: (6615000,) of duration: 300.0 seconds
completed adding noise to data


## FEATURE EXTRACTION - no noise

In [6]:

s2g_laplacian_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='laplacian', quantization_levels=laplacian_quantization_levels)
s2g_wasserstein_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='wasserstein', quantization_levels=wasserstein_quantization_levels)
correlation_detector = ut.ClassicDetector(mode="correlation", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
entropy_detector = ut.ClassicDetector(mode="entropy", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
demon_detector = ut.ClassicDetector(mode="demon", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)

F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

demon_scores = []
correlation_scores = []
entropy_scores = []
laplacian_s2g_scores = []
wasserstein_s2g_scores = []

step_size = 10 * fs
for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    # demon
    demon, F = demon_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = demon[fix]
    demon_scores.append(score)

    # correlation
    correlation, F = correlation_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = correlation[fix]
    correlation_scores.append(score)

    # entropy
    entropy, F = entropy_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = entropy[fix]
    entropy_scores.append(score)

    # s2g score - laplacian
    laplacian, F = s2g_laplacian_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = laplacian[fix]
    laplacian_s2g_scores.append(score)

    # s2g score - wasserstein
    wasserstein, F = s2g_wasserstein_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = wasserstein[fix]
    wasserstein_s2g_scores.append(score)

# normalize scores
demon_scores = (np.array(demon_scores) - np.min(np.array(demon_scores))) / np.std(np.array(demon_scores))
correlation_scores = (np.array(correlation_scores) - np.min(np.array(correlation_scores))) / np.std(np.array(correlation_scores))
entropy_scores = (np.array(entropy_scores) - np.min(np.array(entropy_scores))) / np.std(np.array(entropy_scores))
laplacian_s2g_scores = (np.array(laplacian_s2g_scores) - np.min(np.array(laplacian_s2g_scores))) / np.std(np.array(laplacian_s2g_scores))
wasserstein_s2g_scores = (np.array(wasserstein_s2g_scores) - np.min(np.array(wasserstein_s2g_scores))) / np.std(np.array(wasserstein_s2g_scores))

detectors_taxis_clean = np.linspace(T[0], T[-1], len(laplacian_s2g_scores))

print("completed all feature extraction - no noise")


completed all feature extraction - no noise


In [7]:
# plot original results with added noise

fig = make_subplots(rows=7,
                    cols=1, 
                    row_heights=[0.4, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.03, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Distance (m)", "Demon", "Correlation", "Entropy", "S2G - Laplacian", "S2G - Wasserstein"])

# update axes
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)
fig.update_yaxes(title_text="Score", row=6, col=1)
fig.update_yaxes(title_text="Score", row=7, col=1)
fig.update_xaxes(title_text="Time (s)", row=7, col=1)
fig.update_layout(height=800, width=1400, margin=dict(r=200))
for ann in fig.layout.annotations:
    ann.font.size = 16
    ann.textangle = 0
    ann.x = 1.02
    ann.xanchor = 'left'

_ds = max(1, Sxx.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx[:, ::_ds], x=T[::_ds], y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
t_turn = np.where(T >= 1100)[0][0]
ix_turn = int((t_turn / len(T)) * len(detectors_taxis_clean))


fig.add_trace(go.Scatter(x=detectors_taxis_clean, y=demon_scores, name=f'Demon f={f} Hz', marker_color='blue', showlegend=False), row=3, col=1)
th = np.max(demon_scores[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=3, col=1)
dix = np.where(detectors_taxis_clean >= 865)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_clean))]
_vline_x = detectors_taxis_clean[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=3, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=3, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_clean, y=correlation_scores, name=f'Correlation f={f} Hz', marker_color='blue', showlegend=False), row=4, col=1)
th = np.max(correlation_scores[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=4, col=1)
dix = np.where(detectors_taxis_clean >= 834)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_clean))]
_vline_x = detectors_taxis_clean[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=4, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=4, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_clean, y=entropy_scores, name=f'Entropy f={f} Hz', marker_color='blue', showlegend=False), row=5, col=1)
th = np.max(entropy_scores[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=5, col=1)
dix = np.where(detectors_taxis_clean >= 1100)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_clean))]
_vline_x = detectors_taxis_clean[dix]
# fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=5, col=1)
# fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
#                    showarrow=False, yanchor='bottom', xanchor='center', 
#                    font=dict(size=16, color='red'), row=5, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_clean, y=laplacian_s2g_scores, name=f'Score f={f} Hz (Laplacian)', marker_color='green', showlegend=False), row=6, col=1)
th = np.max(laplacian_s2g_scores[ix_turn:])  # 3.5
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=6, col=1)
dix = np.where(detectors_taxis_clean >= 762)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_clean))]
_vline_x = detectors_taxis_clean[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=6, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=6, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_clean, y=wasserstein_s2g_scores, name=f'Score f={f} Hz (Wasserstein)', marker_color='green', showlegend=False), row=7, col=1)
th = np.max(wasserstein_s2g_scores[ix_turn:])  # 4.0
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=7, col=1)
dix = np.where(detectors_taxis_clean >= 51)[0][0] 
_distance = distances[int(len(distances) * dix / len(detectors_taxis_clean))]
_vline_x = detectors_taxis_clean[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=7, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=7, col=1)

if to_show:
    fig.show(renderer="browser")

In [8]:
if to_save:
    fig.write_image(os.path.join(output_folder, f"{output_file_name}_clean.pdf"))

print("completed plotting original results without noise")

completed plotting original results without noise


## FEATURE EXTRACTION - signal with noise

In [9]:
F_n, T_n, Sxx_n, phasogram_n = ut.calc_spectrogram(noisy_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

s2g_laplacian_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='laplacian', quantization_levels=laplacian_quantization_levels)
s2g_wasserstein_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='wasserstein', quantization_levels=wasserstein_quantization_levels)
correlation_detector = ut.ClassicDetector(mode="correlation", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
entropy_detector = ut.ClassicDetector(mode="entropy", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
demon_detector = ut.ClassicDetector(mode="demon", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)

demon_scores_n = []
correlation_scores_n = []
entropy_scores_n = []
laplacian_s2g_scores_n = []
wasserstein_s2g_scores_n = []

step_size = 10 * fs
for si in np.arange(start=0, stop=(len(noisy_data)-60*fs), step=step_size):
    _data = noisy_data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    # demon
    demon, F = demon_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = demon[fix]
    demon_scores_n.append(score)

    # correlation
    correlation, F = correlation_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = correlation[fix]
    correlation_scores_n.append(score)

    # entropy
    entropy, F = entropy_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = entropy[fix]
    entropy_scores_n.append(score)

    # s2g score - laplacian
    laplacian, F = s2g_laplacian_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = laplacian[fix]
    laplacian_s2g_scores_n.append(score)

    # s2g score - wasserstein
    wasserstein, F = s2g_wasserstein_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = wasserstein[fix]
    wasserstein_s2g_scores_n.append(score)

detectors_taxis_noisy = np.linspace(T_n[0], T_n[-1], len(laplacian_s2g_scores_n))

# normalize scores
demon_scores_n = (np.array(demon_scores_n) - np.min(np.array(demon_scores_n))) / np.std(np.array(demon_scores_n))
correlation_scores_n = (np.array(correlation_scores_n) - np.min(np.array(correlation_scores_n))) / np.std(np.array(correlation_scores_n))
entropy_scores_n = (np.array(entropy_scores_n) - np.min(np.array(entropy_scores_n))) / np.std(np.array(entropy_scores_n))
laplacian_s2g_scores_n = (np.array(laplacian_s2g_scores_n) - np.min(np.array(laplacian_s2g_scores_n))) / np.std(np.array(laplacian_s2g_scores_n))
wasserstein_s2g_scores_n = (np.array(wasserstein_s2g_scores_n) - np.min(np.array(wasserstein_s2g_scores_n))) / np.std(np.array(wasserstein_s2g_scores_n))

print("completed all feature extraction - with noise")

completed all feature extraction - with noise


In [10]:
# plot original results with added noise

fig = make_subplots(rows=7,
                    cols=1, 
                    row_heights=[0.4, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.03, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Distance (m)", "Demon", "Correlation", "Entropy", "S2G - Laplacian", "S2G - Wasserstein"])

# update axes
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)
fig.update_yaxes(title_text="Score", row=6, col=1)
fig.update_yaxes(title_text="Score", row=7, col=1)
fig.update_xaxes(title_text="Time (s)", row=7, col=1)
fig.update_layout(height=800, width=1400, margin=dict(r=200))
for ann in fig.layout.annotations:
    ann.font.size = 16
    ann.textangle = 0
    ann.x = 1.02
    ann.xanchor = 'left'

_ds = max(1, Sxx_n.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx_n[:, ::_ds], x=T_n[::_ds], y=F_n, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
t_turn = np.where(T_n >= 1100)[0][0]
ix_turn = int((t_turn / len(T_n)) * len(detectors_taxis_noisy))


fig.add_trace(go.Scatter(x=detectors_taxis_noisy, y=demon_scores_n, name=f'Demon f={f} Hz', marker_color='blue', showlegend=False), row=3, col=1)
th = np.max(demon_scores_n[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=3, col=1)
dix = np.where(detectors_taxis_noisy >= 865)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_noisy))]
_vline_x = detectors_taxis_noisy[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=3, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=3, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_noisy, y=correlation_scores_n, name=f'Correlation f={f} Hz', marker_color='blue', showlegend=False), row=4, col=1)
th = np.max(correlation_scores_n[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=4, col=1)
dix = np.where(detectors_taxis_noisy >= 824)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_noisy))]
_vline_x = detectors_taxis_noisy[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=4, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=4, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_noisy, y=entropy_scores_n, name=f'Entropy f={f} Hz', marker_color='blue', showlegend=False), row=5, col=1)
th = np.max(entropy_scores_n[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=5, col=1)
dix = np.where(detectors_taxis_noisy >= 917)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_noisy))]
_vline_x = detectors_taxis_noisy[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=5, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=5, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_noisy, y=laplacian_s2g_scores_n, name=f'Score f={f} Hz (Laplacian)', marker_color='green', showlegend=False), row=6, col=1)
th = np.max(laplacian_s2g_scores_n[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=6, col=1)
dix = np.where(detectors_taxis_noisy >= 783)[0][0]
_distance = distances[int(len(distances) * dix / len(detectors_taxis_noisy))]
_vline_x = detectors_taxis_noisy[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=6, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=6, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_noisy, y=wasserstein_s2g_scores_n, name=f'Score f={f} Hz (Wasserstein)', marker_color='green', showlegend=False), row=7, col=1)
th = np.max(wasserstein_s2g_scores_n[ix_turn:])
fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=7, col=1)
dix = np.where(detectors_taxis_noisy >= 628)[0][0] 
_distance = distances[int(len(distances) * dix / len(detectors_taxis_noisy))]
_vline_x = detectors_taxis_noisy[dix]
fig.add_vline(x=_vline_x, line_dash="dash", line_color="red", line_width=2, row=7, col=1)
fig.add_annotation(x=_vline_x, y=1, yref='y domain', text=f'{_distance:.0f} m',
                   showarrow=False, yanchor='bottom', xanchor='center', 
                   font=dict(size=16, color='red'), row=7, col=1)

if to_show:
    fig.show(renderer="browser")

In [11]:
if to_save:
    fig.write_image(os.path.join(output_folder, f"{output_file_name}_noisy.pdf"))

print("completed plotting results with noise")

completed plotting results with noise


## FEATURE EXTRACTION - only BG noise

In [12]:
F_bg, T_bg, Sxx_bg, phasogram_bg = ut.calc_spectrogram(bg_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

demon_scores_bg = []
correlation_scores_bg = []
entropy_scores_bg = []
laplacian_s2g_scores_bg = []
wasserstein_s2g_scores_bg = []

step_size = 10 * fs
for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = bg_data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window
    if len(_data) < 60 * fs:
        break

    # demon
    demon, _, _ = ut.demon_spectrum_narrowband(_data, fs, f, 3, remove_dc=dc, crop_freq=crop_freq, agg='max')
    demon_scores_bg.append(demon)

    # correlation
    TD = talmon_detector.get_score_1fb(_data, f)
    correlation_scores_bg.append(TD)

    # entropy
    entropy, _, _ = ut.spectral_entropy_narrowband(_data, fs, nperseg, overlap, window, target_freq=f, bandwidth=3, remove_dc=dc, crop_freq=crop_freq, agg='max')
    entropy_scores_bg.append(entropy)

    # s2g score - laplacian
    K = s2g_laplacian_detector.get_score_1fb(_data, f)
    laplacian_s2g_scores_bg.append(K)

    # s2g score - wasserstein
    K = s2g_wasserstein_detector.get_score_1fb(_data, f)
    wasserstein_s2g_scores_bg.append(K)

detectors_taxis_bg = np.linspace(T_bg[0], T_bg[-1], len(laplacian_s2g_scores_bg))

print("completed all feature extraction - only BG noise")


NameError: name 'talmon_detector' is not defined

In [ ]:

s2g_laplacian_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='laplacian', quantization_levels=laplacian_quantization_levels)
s2g_wasserstein_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='wasserstein', quantization_levels=wasserstein_quantization_levels)
correlation_detector = ut.ClassicDetector(mode="correlation", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
entropy_detector = ut.ClassicDetector(mode="entropy", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
demon_detector = ut.ClassicDetector(mode="demon", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)

F_bg, T_bg, Sxx_bg, phasogram_bg = ut.calc_spectrogram(bg_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

demon_scores_bg = []
correlation_scores_bg = []
entropy_scores_bg = []
laplacian_s2g_scores_bg = []
wasserstein_s2g_scores_bg = []

step_size = 10 * fs
for si in np.arange(start=0, stop=(len(bg_data)-60*fs), step=step_size):
    _data = bg_data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window
    if len(_data) < 60 * fs:
        break

    # demon
    demon, F = demon_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = demon[fix]
    demon_scores_bg.append(score)

    # correlation
    correlation, F = correlation_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = correlation[fix]
    correlation_scores_bg.append(score)

    # entropy
    entropy, F = entropy_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = entropy[fix]
    entropy_scores_bg.append(score)

    # s2g score - laplacian
    laplacian, F = s2g_laplacian_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = laplacian[fix]
    laplacian_s2g_scores_bg.append(score)

    # s2g score - wasserstein
    wasserstein, F = s2g_wasserstein_detector.get_feature_vector(_data)
    fix = np.where(F >= f)[0][0]
    score = wasserstein[fix]
    wasserstein_s2g_scores_bg.append(score)

detectors_taxis_bg = np.linspace(T_bg[0], T_bg[-1], len(laplacian_s2g_scores_bg))

# normalize scores
demon_scores_bg = (np.array(demon_scores_bg) - np.min(np.array(demon_scores_bg))) / np.std(np.array(demon_scores_bg))
correlation_scores_bg = (np.array(correlation_scores_bg) - np.min(np.array(correlation_scores_bg))) / np.std(np.array(correlation_scores_bg))
entropy_scores_bg = (np.array(entropy_scores_bg) - np.min(np.array(entropy_scores_bg))) / np.std(np.array(entropy_scores_bg))
laplacian_s2g_scores_bg = (np.array(laplacian_s2g_scores_bg) - np.min(np.array(laplacian_s2g_scores_bg))) / np.std(np.array(laplacian_s2g_scores_bg))
wasserstein_s2g_scores_bg = (np.array(wasserstein_s2g_scores_bg) - np.min(np.array(wasserstein_s2g_scores_bg))) / np.std(np.array(wasserstein_s2g_scores_bg))

print("completed all feature extraction - only BG noise")

KeyboardInterrupt: 

In [ ]:
# plot original results with added noise

fig = make_subplots(rows=6,
                    cols=1, 
                    row_heights=[0.4, 0.1, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Demon", "Correlation", "Entropy", "S2G - Laplacian", "S2G - Wasserstein"])


_ds = max(1, Sxx_bg.shape[1] // 1000)
fig.add_trace(go.Heatmap(z=Sxx_bg[:, ::_ds], x=T_bg[::_ds], y=F_bg, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
# t_turn = np.where(T_bg >= 1100)[0][0]
# ix_turn = int((t_turn / len(T_bg)) * len(detectors_taxis_bg))
# print(f"t_turn: {t_turn}, ix_turn: {ix_turn}")


fig.add_trace(go.Scatter(x=detectors_taxis_bg, y=demon_scores_bg, name=f'Demon f={f} Hz', marker_color='blue', showlegend=False), row=2, col=1)
# th = np.max(demon_scores_bg[ix_turn:])
# fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=2, col=1)
# dix = np.where(detectors_taxis_clean >= 443)[0][0]
# fig.add_vline(x=detectors_taxis_clean[dix], line_dash="dash", line_color="red", line_width=2, row=3, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_bg, y=correlation_scores_bg, name=f'Correlation f={f} Hz', marker_color='blue', showlegend=False), row=3, col=1)
# th = np.max(correlation_scores_bg[ix_turn:])
# fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=3, col=1)
# dix = np.where(detectors_taxis_clean >= 381)[0][0]
# fig.add_vline(x=detectors_taxis_clean[dix], line_dash="dash", line_color="red", line_width=2, row=4, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_bg, y=entropy_scores_bg, name=f'Entropy f={f} Hz', marker_color='blue', showlegend=False), row=4, col=1)
# th = np.max(entropy_scores_bg[ix_turn:])
# fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=4, col=1)
# dix = np.where(detectors_taxis_clean >= 917)[0][0]
# fig.add_vline(x=detectors_taxis_clean[dix], line_dash="dash", line_color="red", line_width=2, row=5, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_bg, y=laplacian_s2g_scores_bg, name=f'Score f={f} Hz (Laplacian)', marker_color='green', showlegend=False), row=5, col=1)
# th = np.max(laplacian_s2g_scores_bg[ix_turn:])
# fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=6, col=1)
# dix = np.where(detectors_taxis_clean >= 720)[0][0]
# fig.add_vline(x=detectors_taxis_clean[dix], line_dash="dash", line_color="red", line_width=2, row=6, col=1)

fig.add_trace(go.Scatter(x=detectors_taxis_bg, y=wasserstein_s2g_scores_bg, name=f'Score f={f} Hz (Wasserstein)', marker_color='green', showlegend=False), row=6, col=1)
# # th = np.max(wasserstein_s2g_scores_bg[ix_turn:])
# fig.add_hline(y=th, line_dash="dash", line_color="black", line_width=1, row=7, col=1)
# dix = np.where(detectors_taxis_clean >= 20)[0][0] 
# fig.add_vline(x=detectors_taxis_clean[dix], line_dash="dash", line_color="red", line_width=2, row=7, col=1)

# update axes
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Score", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)
fig.update_yaxes(title_text="Score", row=6, col=1)
fig.update_xaxes(title_text="Time (s)", row=6, col=1)
fig.update_layout(height=800, width=1400, margin=dict(r=200))
for ann in fig.layout.annotations:
    ann.font.size = 16
    ann.textangle = 0
    ann.x = 1.02
    ann.xanchor = 'left'

if to_show:
    fig.show(renderer="browser")

In [ ]:
if to_save:
    fig.write_image(os.path.join(output_folder, f"{output_file_name}_bg.pdf"))

print("completed plotting results with BG noise only")

completed plotting results with BG noise only


### DONE.